# Simulación y Demostración: Algoritmo de Atribución UAS
Este notebook ejecuta los 3 escenarios de prueba y el caso borde para identificar el emisor primario.

In [ ]:
import sys, os
sys.path.append('..')

from src.graph_builder import (
    build_star_network,
    build_relay_chain_network,
    build_complex_mesh_network,
    build_disconnected_network
)
from src.attribution_engine import identify_primary_emitter
from src.visualizer import draw_attribution_graph

os.makedirs('../docs/img', exist_ok=True)

## Escenario A: Control Directo Punto a Punto

In [ ]:
G_star = build_star_network()
emitter, dist, candidates = identify_primary_emitter(G_star, "Dron_D1")

print(f"Emisor Primario Identificado: {emitter}")
print(f"Distancia minima de trayectoria: {dist}")
print(f"Todos los candidatos detectados: {candidates}")

plot = draw_attribution_graph(G_star, "Dron_D1", emitter, candidates, "Escenario A: Control Directo")
plot.savefig('../docs/img/escenario_a_directo.png', dpi=150)
plot.show()

## Escenario B: Cadena de Repetidores

In [ ]:
G_chain = build_relay_chain_network()
emitter, dist, candidates = identify_primary_emitter(G_chain, "Dron_D1")

print(f"Emisor Primario Identificado: {emitter}")
print(f"Distancia minima de trayectoria: {dist}")
print(f"Todos los candidatos detectados: {candidates}")

plot = draw_attribution_graph(G_chain, "Dron_D1", emitter, candidates, "Escenario B: Cadena de Repetidores")
plot.savefig('../docs/img/escenario_b_cadena.png', dpi=150)
plot.show()

## Escenario C: Red Mesh Compleja con Múltiples Candidatos (Desempate Matemático)

In [ ]:
G_mesh = build_complex_mesh_network()
emitter, dist, candidates = identify_primary_emitter(G_mesh, "Dron_D1")

print(f"Emisor Primario Identificado: {emitter}")
print(f"Distancia minima de trayectoria: {dist}")
print(f"Todos los candidatos detectados: {candidates}")

plot = draw_attribution_graph(G_mesh, "Dron_D1", emitter, candidates, "Escenario C: Atribucion en Red Mesh")
plot.savefig('../docs/img/escenario_c_mesh.png', dpi=150)
plot.show()

## Caso Borde: Red Desconectada (Atribución No Determinable)

In [ ]:
G_disc = build_disconnected_network()
emitter, dist, candidates = identify_primary_emitter(G_disc, "Dron_Aislado_D1")

print(f"Emisor Primario Identificado: {emitter}")
print(f"Distancia minima de trayectoria: {dist}")
print(f"Todos los candidatos detectados: {candidates}")

## Validación Empírica de la Complejidad Computacional
Se generan redes de tamaño creciente para medir el tiempo de ejecución real de `identify_primary_emitter` y verificar que escala linealmente respecto a $|V| + |E|$, tal como se demostró teóricamente.

In [ ]:
import time
import matplotlib.pyplot as plt
from src.graph_builder import build_layered_random_network

layer_sizes = [5, 10, 20, 40, 80, 160]
nodes_per_layer = 15
sizes = []
times = []

for num_layers in layer_sizes:
    G, target = build_layered_random_network(num_layers, nodes_per_layer)
    n_edges_vertices = G.number_of_nodes() + G.number_of_edges()

    best_time = min(
        (lambda: (lambda t0, r, t1: t1 - t0)(time.perf_counter(), identify_primary_emitter(G, target), time.perf_counter()))()
        for _ in range(5)
    )

    sizes.append(n_edges_vertices)
    times.append(best_time)
    print(f"Capas: {num_layers:>3} | |V|+|E| = {n_edges_vertices:>5} | Tiempo: {best_time*1000:.3f} ms")

plt.figure(figsize=(8, 5))
plt.plot(sizes, times, marker='o', color="#E63946", linewidth=2)
plt.xlabel("Tamaño del grafo (|V| + |E|)")
plt.ylabel("Tiempo de ejecución (segundos)")
plt.title("Validación Empírica: Escalabilidad Lineal O(|V| + |E|)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/img/complejidad_empirica.png', dpi=150)
plt.show()